# Lymphoma Histopathology Classification

Train and evaluate two models (custom CNN + ResNet-18) on H&E-stained
lymphoma patches (CLL, FL, MCL) from the Kaggle dataset.

**Goal**: Validate the classification pipeline before fine-tuning
on institutional MZL vs. reactive lymphadenopathy data.

## 1. Setup

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from models import LymphomaNet, get_resnet18, unfreeze

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Configuration

In [ ]:
# ──── EDIT THESE ────
DATA_DIR    = "./data/lymphoma"    # folder with CLL/, FL/, MCL/ subfolders
OUTPUT_DIR  = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training
IMAGE_SIZE  = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
LR          = 1e-4
PATIENCE    = 7

# Which model to train: "lymphomanet" or "resnet18"
MODEL_NAME  = "resnet18"

CLASS_NAMES = ["CLL", "FL", "MCL"]

## 3. Data Loading

In [ ]:
def load_data(data_dir):
    """Scan folder structure and return paths + labels."""
    paths, labels = [], []
    classes = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)))
    for idx, cls in enumerate(classes):
        files = [os.path.join(data_dir, cls, f) for f in os.listdir(os.path.join(data_dir, cls))
                 if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.jpeg'))]
        paths.extend(files)
        labels.extend([idx] * len(files))
        print(f"  {cls}: {len(files)} images")
    print(f"  Total: {len(paths)}")
    return paths, labels, classes

paths, labels, class_names = load_data(DATA_DIR)

In [ ]:
# Stratified split: 70% train, 15% val, 15% test
train_val_p, test_p, train_val_l, test_l = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=SEED)
train_p, val_p, train_l, val_l = train_test_split(
    train_val_p, train_val_l, test_size=0.176, stratify=train_val_l, random_state=SEED)

print(f"Train: {len(train_p)} | Val: {len(val_p)} | Test: {len(test_p)}")

## 4. Dataset & Augmentation

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[i]

# Histopathology augmentation: any rotation is valid, color jitter = stain variation
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = PatchDataset(train_p, train_l, train_tf)
val_ds   = PatchDataset(val_p, val_l, val_tf)
test_ds  = PatchDataset(test_p, test_l, val_tf)

# Weighted sampler for class imbalance
weights = 1.0 / np.bincount(train_l)
sampler = WeightedRandomSampler([weights[l] for l in train_l], len(train_l))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## 5. Build Model

In [ ]:
if MODEL_NAME == "lymphomanet":
    model = LymphomaNet(num_classes=len(class_names)).to(DEVICE)
else:
    model = get_resnet18(num_classes=len(class_names), freeze=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {trainable:,} / {total:,} trainable params ({100*trainable/total:.0f}%)")

## 6. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    # Unfreeze ResNet after epoch 10
    if MODEL_NAME == "resnet18" and epoch == 10:
        print("  → Unfreezing all layers")
        unfreeze(model)
        optimizer = optim.AdamW(model.parameters(), lr=LR * 0.1, weight_decay=1e-4)

    # ── Train ──
    model.train()
    running_loss, correct, total = 0, 0, 0
    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += lbls.size(0)
    train_loss, train_acc = running_loss / total, correct / total

    # ── Validate ──
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out = model(imgs)
            loss = criterion(out, lbls)
            running_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == lbls).sum().item()
            total += lbls.size(0)
    val_loss, val_acc = running_loss / total, correct / total
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    lr = optimizer.param_groups[0]["lr"]
    print(f"  Epoch {epoch+1:2d} | Train {train_loss:.4f} ({train_acc:.1%}) | Val {val_loss:.4f} ({val_acc:.1%}) | LR {lr:.1e}")

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
        print(f"           → saved (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs, history["train_loss"], "b-", label="Train")
ax1.plot(epochs, history["val_loss"], "r-", label="Val")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, [a*100 for a in history["train_acc"]], "b-", label="Train")
ax2.plot(epochs, [a*100 for a in history["val_acc"]], "r-", label="Val")
ax2.set(xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 8. Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pth"), map_location=DEVICE, weights_only=True))
model.eval()

all_probs, all_labels, all_preds = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE)
        probs = torch.softmax(model(imgs), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(lbls.numpy())
        all_preds.append(probs.argmax(1).cpu().numpy())

all_probs  = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_preds  = np.concatenate(all_preds)

# Metrics
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))
print(f"Macro F1:  {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"Macro AUC: {roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro'):.4f}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(all_labels, all_preds), annot=True, fmt="d",
            cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set(xlabel="Predicted", ylabel="True", title="Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 9. Sample Predictions

In [ ]:
# Show 10 random test images with predictions
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
indices = random.sample(range(len(test_p)), 10)

for ax, idx in zip(axes.flat, indices):
    img = Image.open(test_p[idx]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    true = class_names[test_l[idx]]
    pred = class_names[all_preds[idx]]
    conf = all_probs[idx].max()
    correct = true == pred

    ax.imshow(img)
    color = "#27ae60" if correct else "#e74c3c"
    symbol = "✓" if correct else "✗"
    ax.set_title(f"{symbol} True: {true}\nPred: {pred} ({conf:.0%})", fontsize=10, color=color, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f"Test Predictions ({MODEL_NAME})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=150)
plt.show()

## Next Steps

Fine-tune this model on institutional MZL vs. reactive lymphadenopathy data:
1. Change `DATA_DIR` to point to new data (two folders: `MZL/` and `Reactive/`)
2. Set `CLASS_NAMES = ["MZL", "Reactive"]`
3. Load pretrained weights: `model.load_state_dict(torch.load("results/best_model.pth"))`
4. Replace the classifier head for 2 classes
5. Train with lower LR (1e-5)